# San Francisco Salaries — Employee Profile System

**Highridge / NXU Assignment — Data Processing, Error Handling & File Handling in Python**

This notebook works with the provided San Francisco city employee salary
dataset (`salaries.csv`, 2011–2018) and performs six tasks:

1. Import the provided salary data
2. Create a function that returns an employee's details by name
3. Process the salary data using a Python dictionary
4. Implement error handling throughout
5. Export one employee's details to CSV and zip it into an **"Employee Profile"** folder
6. Use R to unzip that folder and display the data (see `unzip_and_display.R`, included alongside this notebook)

---

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import os
import csv
import shutil
import zipfile

pd.set_option("display.max_columns", None)

## 1. Import Data

The salary data is read from `salaries.csv` (the provided dataset). The
read is wrapped in error handling in case the file is missing, empty, or
has an unexpected encoding — all common real-world issues with CSV
imports.

The four pay columns (`BasePay`, `OvertimePay`, `OtherPay`, `Benefits`)
sometimes contain non-numeric placeholder text (e.g. `"Not Provided"`) in
the raw data, so they're coerced to numeric, turning any unparsable
values into `NaN` rather than crashing the import.

In [2]:
DATA_PATH = "salaries.csv"

def load_salary_data(path=DATA_PATH):
    """
    Loads the salary dataset with error handling for common file issues:
    missing file, empty file, and encoding problems.
    """
    try:
        data = pd.read_csv(path, low_memory=False)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"Could not find '{path}'. Make sure salaries.csv is in the "
            "same folder as this notebook."
        )
    except pd.errors.EmptyDataError:
        raise ValueError(f"'{path}' was found but contains no data.")
    except UnicodeDecodeError:
        # Fall back to a more permissive encoding if UTF-8 fails
        data = pd.read_csv(path, low_memory=False, encoding="latin1")

    if data.empty:
        raise ValueError(f"'{path}' loaded but the resulting DataFrame is empty.")

    # Normalize the pay columns to numeric; bad values become NaN instead
    # of raising an error and stopping the whole import.
    for column in ["BasePay", "OvertimePay", "OtherPay", "Benefits"]:
        if column in data.columns:
            data[column] = pd.to_numeric(data[column], errors="coerce")

    return data


df = load_salary_data()
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns.")
df.head()

Loaded 148,654 rows and 13 columns.


,Id,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,TotalPay,TotalPayBenefits,Year,Notes,Agency,Status
0,1,NATHANIEL FORD,GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY,167411.18,0.00,400184.25,NaN,567595.43,567595.43,2011,NaN,San Francisco,NaN
1,2,GARY JIMENEZ,CAPTAIN III (POLICE DEPARTMENT),155966.02,245131.88,137811.38,NaN,538909.28,538909.28,2011,NaN,San Francisco,NaN
2,3,ALBERT PARDINI,CAPTAIN III (POLICE DEPARTMENT),212739.13,106088.18,16452.60,NaN,335279.91,335279.91,2011,NaN,San Francisco,NaN
3,4,CHRISTOPHER CHONG,WIRE ROPE CABLE MAINTENANCE MECHANIC,77916.00,56120.71,198306.90,NaN,332343.61,332343.61,2011,NaN,San Francisco,NaN
4,5,PATRICK GARDNER,"DEPUTY CHIEF OF DEPARTMENT,(FIRE DEPARTMENT)",134401.60,9737.00,182234.59,NaN,326373.19,326373.19,2011,NaN,San Francisco,NaN


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 148654 entries, 0 to 148653
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Id                148654 non-null  int64  
 1   EmployeeName      148654 non-null  str    
 2   JobTitle          148654 non-null  str    
 3   BasePay           148045 non-null  float64
 4   OvertimePay       148650 non-null  float64
 5   OtherPay          148650 non-null  float64
 6   Benefits          112491 non-null  float64
 7   TotalPay          148654 non-null  float64
 8   TotalPayBenefits  148654 non-null  float64
 9   Year              148654 non-null  int64  
 10  Notes             0 non-null       float64
 11  Agency            148654 non-null  str    
 12  Status            38119 non-null   str    
dtypes: float64(7), int64(2), str(4)
memory usage: 14.7 MB


## 2. Data Processing with a Dictionary

Employee names are **not unique** in this dataset — the same person can
appear once per year they were employed. To make employee lookups fast
and to satisfy the "process the data using a dictionary" requirement, the
DataFrame is converted into a dictionary keyed by (uppercased, stripped)
employee name, where each value is a **list of yearly records** for that
person.

In [4]:
def build_employee_dictionary(data):
    """
    Converts the salary DataFrame into a dictionary of the form:
        { "EMPLOYEE NAME": [ {year 1 record}, {year 2 record}, ... ], ... }
    Wrapped in error handling in case the expected columns are missing.
    """
    required_columns = {
        "EmployeeName", "JobTitle", "BasePay", "OvertimePay", "OtherPay",
        "Benefits", "TotalPay", "TotalPayBenefits", "Year", "Agency",
    }
    missing = required_columns - set(data.columns)
    if missing:
        raise KeyError(f"Dataset is missing expected column(s): {missing}")

    employee_dict = {}
    try:
        for row in data.to_dict(orient="records"):
            key = str(row["EmployeeName"]).strip().upper()
            record = {
                "JobTitle": row.get("JobTitle"),
                "BasePay": row.get("BasePay"),
                "OvertimePay": row.get("OvertimePay"),
                "OtherPay": row.get("OtherPay"),
                "Benefits": row.get("Benefits"),
                "TotalPay": row.get("TotalPay"),
                "TotalPayBenefits": row.get("TotalPayBenefits"),
                "Year": row.get("Year"),
                "Agency": row.get("Agency"),
            }
            employee_dict.setdefault(key, []).append(record)
    except Exception as err:
        raise RuntimeError(f"Failed to build employee dictionary: {err}")

    return employee_dict


employee_dict = build_employee_dictionary(df)
print(f"Dictionary built for {len(employee_dict):,} unique employee names.")
# Peek at one entry
sample_key = "NATHANIEL FORD"
employee_dict.get(sample_key)

Dictionary built for 80,459 unique employee names.


[{'JobTitle': 'GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY',
  'BasePay': 167411.18,
  'OvertimePay': 0.0,
  'OtherPay': 400184.25,
  'Benefits': nan,
  'TotalPay': 567595.43,
  'TotalPayBenefits': 567595.43,
  'Year': 2011,
  'Agency': 'San Francisco'}]

## 3. Create Employee Function

`get_employee_details()` accepts an employee's name and returns their
record(s) from the dictionary built above. A custom exception,
`EmployeeNotFoundError`, is used so callers can catch "not found" cases
specifically and distinguish them from programming errors (e.g. passing
in a non-string name).

In [5]:
class EmployeeNotFoundError(Exception):
    """Raised when a requested employee name isn't in the dataset."""
    pass


def get_employee_details(name, data=employee_dict):
    """
    Looks up an employee by name (case-insensitive, whitespace-tolerant)
    and returns a list of their yearly salary records.

    Raises:
        TypeError: if `name` isn't a string.
        ValueError: if `name` is blank.
        EmployeeNotFoundError: if no employee matches `name`.
    """
    if not isinstance(name, str):
        raise TypeError(f"Employee name must be a string, got {type(name).__name__}.")

    clean_name = name.strip().upper()
    if not clean_name:
        raise ValueError("Employee name cannot be blank.")

    if clean_name not in data:
        raise EmployeeNotFoundError(f"No employee found matching '{name}'.")

    return data[clean_name]

## 4. Error Handling in Action\n\nA few example calls showing the function behaving well on good input and failing *gracefully* on bad input.

In [6]:
# 1) A normal, successful lookup
try:
    details = get_employee_details("NATHANIEL FORD")
    print(f"Found {len(details)} record(s) for NATHANIEL FORD:")
    for record in details:
        print(f"  {record['Year']}: {record['JobTitle']} — "
              f"${record['TotalPayBenefits']:,.2f} total pay & benefits")
except (TypeError, ValueError, EmployeeNotFoundError) as err:
    print(f"Error: {err}")

Found 1 record(s) for NATHANIEL FORD:
  2011: GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY — $567,595.43 total pay & benefits


In [7]:
# 2) An employee who doesn't exist in the dataset
try:
    get_employee_details("JOHN Q. NOBODY")
except EmployeeNotFoundError as err:
    print(f"Handled gracefully -> {err}")

Handled gracefully -> No employee found matching 'JOHN Q. NOBODY'.


In [8]:
# 3) Bad input type (not a string at all)
try:
    get_employee_details(12345)
except TypeError as err:
    print(f"Handled gracefully -> {err}")

Handled gracefully -> Employee name must be a string, got int.


In [9]:
# 4) Blank input
try:
    get_employee_details("   ")
except ValueError as err:
    print(f"Handled gracefully -> {err}")

Handled gracefully -> Employee name cannot be blank.


## 5. Export Employee Details to CSV, Zipped into "Employee Profile"

`export_employee_profile()` looks up an employee, writes their records to
a CSV file inside a folder named **`Employee Profile`**, then compresses
that entire folder into `Employee Profile.zip`. Every file operation is
wrapped in error handling so a permissions issue or disk problem doesn't
crash the notebook.

In [10]:
def export_employee_profile(name, data=employee_dict, folder_name="Employee Profile"):
    """
    Exports one employee's salary records to a CSV file inside `folder_name`,
    then zips that folder into `<folder_name>.zip`.

    Returns the path to the resulting zip file, or None if export failed.
    """
    # Look up the employee first — no point creating folders/files if this fails
    try:
        records = get_employee_details(name, data)
    except (TypeError, ValueError, EmployeeNotFoundError) as err:
        print(f"Could not export profile: {err}")
        return None

    try:
        os.makedirs(folder_name, exist_ok=True)
    except OSError as err:
        print(f"Could not create folder '{folder_name}': {err}")
        return None

    safe_filename = name.strip().upper().replace(" ", "_") + "_profile.csv"
    csv_path = os.path.join(folder_name, safe_filename)

    fieldnames = [
        "EmployeeName", "Year", "JobTitle", "BasePay", "OvertimePay",
        "OtherPay", "Benefits", "TotalPay", "TotalPayBenefits", "Agency",
    ]

    try:
        with open(csv_path, "w", newline="", encoding="utf-8") as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
            writer.writeheader()
            for record in records:
                row = {"EmployeeName": name.strip().upper(), **record}
                writer.writerow(row)
    except (OSError, csv.Error) as err:
        print(f"Could not write CSV file: {err}")
        return None

    zip_base_name = folder_name  # shutil appends .zip
    try:
        zip_path_no_ext = shutil.make_archive(
            base_name=zip_base_name, format="zip", root_dir=".", base_dir=folder_name
        )
    except (OSError, shutil.Error) as err:
        print(f"Could not create zip archive: {err}")
        return None

    print(f"Exported {len(records)} record(s) for {name.strip().upper()} -> {csv_path}")
    print(f"Zipped '{folder_name}/' -> {zip_path_no_ext}")
    return zip_path_no_ext

In [11]:
zip_path = export_employee_profile("NATHANIEL FORD")

# Confirm what actually landed inside the zip
if zip_path:
    with zipfile.ZipFile(zip_path) as zf:
        print("\nContents of the zip file:")
        for name in zf.namelist():
            print(" ", name)

Exported 1 record(s) for NATHANIEL FORD -> Employee Profile/NATHANIEL_FORD_profile.csv
Zipped 'Employee Profile/' -> /home/claude/notebook_project/Employee Profile.zip

Contents of the zip file:
  Employee Profile/
  Employee Profile/NATHANIEL_FORD_profile.csv


In [12]:
# Error handling also applies to export — an unknown employee is
# handled gracefully instead of crashing:
export_employee_profile("NOT A REAL PERSON")

Could not export profile: No employee found matching 'NOT A REAL PERSON'.


## 6. Unzip and Display the Data — in R

Steps 1–5 above are all Python, run in this notebook. Step 6 asks for R
specifically, so it's provided as a **standalone R script**,
`unzip_and_display.R`, included alongside this notebook in the
submission. Run it with:

```bash
Rscript unzip_and_display.R
```

The script's logic (shown here for reference) unzips `Employee
Profile.zip` and reads/prints the CSV inside it using base R:

```r
zip_path <- "Employee Profile.zip"
extract_dir <- "Employee Profile"

unzip(zip_path, exdir = ".")

csv_files <- list.files(extract_dir, pattern = "\\.csv$", full.names = TRUE)

for (csv_file in csv_files) {
  employee_data <- read.csv(csv_file, stringsAsFactors = FALSE)
  print(employee_data)
}
```

See `unzip_and_display.R` for the full version with error handling
(missing zip file, empty folder, unreadable CSV, etc.) and
`README.md` for run instructions.